<a href="https://colab.research.google.com/github/IndiraTejaswini/Pytorch_Neural_networks/blob/main/DCGAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#%matplotlib inline
import argparse
import os
import random
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.optim as optim
import torch.utils.data
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torchvision.utils as vutils
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

# Set random seed for reproducibility
manualSeed = 999
#manualSeed = random.randint(1, 10000) # use if you want new results
print("Random Seed: ", manualSeed)
random.seed(manualSeed)
torch.manual_seed(manualSeed)
torch.use_deterministic_algorithms(True) # Needed for reproducible results

Random Seed:  999


**Breakdown of the Code**


manualSeed = 999: Defines a specific starting point for the random number generator. As long as you use this same number, the "random" sequence will be identical every time.

random.seed(...): Sets the seed for Python's built-in random module (used for shuffling data).

torch.manual_seed(...): Sets the seed for PyTorch’s random number generator (used for initializing weights and other operations).

torch.use_deterministic_algorithms(True): This is a strong constraint that forces PyTorch to use algorithms that are guaranteed to be deterministic. Without this, some operations (like certain GPU-accelerated convolutions) might use non-deterministic implementations to gain speed, which would introduce slight variations even if the seed is set.

By fixing these, you create a controlled environment where the only variables changing are the ones you intentionally modify.

dataroot - the path to the root of the dataset folder. We will talk more about the dataset in the next section.

workers - the number of worker threads for loading the data with the DataLoader.

batch_size - the batch size used in training. The DCGAN paper uses a batch size of 128.

image_size - the spatial size of the images used for training. This implementation defaults to 64x64. If another size is desired, the structures of D and G must be changed. See here for more details.

nc - number of color channels in the input images. For color images this is 3.

nz - length of latent vector.

ngf - relates to the depth of feature maps carried through the generator.

ndf - sets the depth of feature maps propagated through the discriminator.

num_epochs - number of training epochs to run. Training for longer will probably lead to better results but will also take much longer.

lr - learning rate for training. As described in the DCGAN paper, this number should be 0.0002.

beta1 - beta1 hyperparameter for Adam optimizers. As described in paper, this number should be 0.5.

ngpu - number of GPUs available. If this is 0, code will run in CPU mode. If this number is greater than 0 it will run on that number of GPUs.

In the context of deep learning models like DCGANs, the "depth" refers to the number of feature maps (often called channels or filters) at each layer of the neural network.

In [ ]:
# Root directory for dataset
dataroot = "data/celeba"

# Number of workers for dataloader
workers = 2

# Batch size during training
batch_size = 128

# Spatial size of training images. All images will be resized to this
#   size using a transformer.
image_size = 64

# Number of channels in the training images. For color images this is 3
nc = 3

# Size of z latent vector (i.e. size of generator input)
nz = 100

# Size of feature maps in generator
ngf = 64

# Size of feature maps in discriminator
ndf = 64

# Number of training epochs
num_epochs = 5

# Learning rate for optimizers
lr = 0.0002

# Beta1 hyperparameter for Adam optimizers
beta1 = 0.5

# Number of GPUs available. Use 0 for CPU mode.
ngpu = 1

In [ ]:
/path/to/celeba
    -> img_align_celeba
        -> 188242.jpg
        -> 173822.jpg
        -> 284702.jpg
        -> 537394.jpg
           ...

IndentationError: unexpected indent (2760091483.py, line 2)

In [ ]:
# We can use an image folder dataset the way we have it setup.
# Create the dataset
dataset = dset.ImageFolder(root=dataroot,
                           transform=transforms.Compose([
                               transforms.Resize(image_size),
                               transforms.CenterCrop(image_size),
                               transforms.ToTensor(),
                               transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
                           ]))
# Create the dataloader
dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size,
                                         shuffle=True, num_workers=workers)

# Decide which device we want to run on
device = torch.device("cuda:0" if (torch.cuda.is_available() and ngpu > 0) else "cpu")

# Plot some training images
real_batch = next(iter(dataloader))
plt.figure(figsize=(8,8))
plt.axis("off")
plt.title("Training Images")
plt.imshow(np.transpose(vutils.make_grid(real_batch[0].to(device)[:64], padding=2, normalize=True).cpu(),(1,2,0)))
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/data/celeba'

### Verify `dataroot` contents

Let's check what's actually inside the `dataroot` path to confirm the dataset structure. The `ImageFolder` expects subdirectories (which it treats as classes). For CelebA, if your images are in `img_align_celeba` inside `celeba`, `dataroot` should point to the `celeba` folder, and `img_align_celeba` will be seen as the 'class' folder. If `img_align_celeba` is directly at the root of your dataset, then `dataroot` should point to `img_align_celeba`.

In [ ]:
import os

# Check if the dataroot directory exists
if not os.path.exists(dataroot):
    print(f"Error: The directory '{dataroot}' does not exist. Please check your Google Drive path.")
else:
    print(f"Contents of '{dataroot}':")
    try:
        for item in os.listdir(dataroot):
            item_path = os.path.join(dataroot, item)
            if os.path.isdir(item_path):
                print(f"  [DIR] {item}")
            else:
                print(f"  [FILE] {item}")
    except Exception as e:
        print(f"Could not list contents of '{dataroot}': {e}")

# If dataroot contains 'img_align_celeba', let's check inside it
possible_img_dir = os.path.join(dataroot, 'img_align_celeba')
if os.path.isdir(possible_img_dir):
    print(f"\nContents of '{possible_img_dir}' (first 10 items):")
    try:
        for i, item in enumerate(os.listdir(possible_img_dir)):
            if i >= 10: break
            item_path = os.path.join(possible_img_dir, item)
            if os.path.isdir(item_path):
                print(f"  [DIR] {item}")
            else:
                print(f"  [FILE] {item}")
        if len(os.listdir(possible_img_dir)) > 10:
            print("  ...")
    except Exception as e:
        print(f"Could not list contents of '{possible_img_dir}': {e}")

Error: The directory '/content/drive/MyDrive/data/celeba' does not exist. Please check your Google Drive path.


### Mount Google Drive

Before running the next cell, please ensure you have uploaded the `img_align_celeba` folder (after extracting from the zip file) to your Google Drive. A recommended structure is `MyDrive/data/celeba/img_align_celeba`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Update `dataroot`

Now, let's update the `dataroot` variable to point to the correct location in your Google Drive. You might need to adjust the path based on where you placed the `celeba` folder. The `ImageFolder` expects the root directory to contain subdirectories that are the classes. For CelebA, typically you'd point it to the parent directory containing `img_align_celeba`. For instance, if `img_align_celeba` is directly inside `celeba`, then `dataroot` should point to the `celeba` folder.

In [ ]:
# Update dataroot to point to your dataset in Google Drive
# IMPORTANT: Adjust this path if your 'celeba' folder is located elsewhere.
# For example, if you uploaded it directly to MyDrive, it might be '/content/drive/MyDrive/celeba'
# Or if it's in a subfolder like 'my_data', then '/content/drive/MyDrive/my_data/celeba'
dataroot = '/content/drive/MyDrive/data/celeba'
print(f"Dataroot set to: {dataroot}")

Dataroot set to: /content/drive/MyDrive/data/celeba
